# Notebook 03 — Agent Trace & Debug
Run the full LangGraph agent on individual queries, inspect intermediate state at each node, and verify SQLite logging.

In [ ]:
import sys, uuid
sys.path.insert(0, '../src')

from agent import compile_app, run_query, AGENT_LOGS_DB
print('Agent modules loaded.')
print('Logs DB:', AGENT_LOGS_DB)

## 1. Compile the agent graph

In [ ]:
app = compile_app()
print('Graph compiled successfully.')

# Draw the graph (requires graphviz)
try:
    from IPython.display import Image
    Image(app.get_graph().draw_mermaid_png())
except Exception as e:
    print('Graph visualisation unavailable:', e)
    print(app.get_graph().draw_mermaid())

## 2. Run a single query and inspect the final state

In [ ]:
THREAD_ID = str(uuid.uuid4())

# --- Change these to test different queries ---
QUERY       = 'What was Apple total revenue in fiscal year 2023?'
TICKER      = 'AAPL'
FILING_TYPE = '10-K'
YEAR        = 2023
# -----------------------------------------------

result = run_query(
    query=QUERY,
    thread_id=THREAD_ID,
    company=TICKER,
    filing_type=FILING_TYPE,
    year=YEAR,
    app=app,
)

print('=== FINAL STATE ===')
print(f'Answer      : {result["answer"][:500]}')
print(f'Retry count : {result["retry_count"]}')
print(f'Validated   : {result["validated"]}')
print(f'Errors      : {result["errors"]}')
print(f'Metrics     : {result["extracted_metrics"]}')

## 3. Trace intermediate messages from each node

In [ ]:
print('=== NODE MESSAGE TRACE ===')
for msg in result.get('messages', []):
    role    = msg.get('role', 'unknown').upper()
    content = msg.get('content', '')[:300]
    print(f'[{role:12s}] {content}')
    print()

## 4. Inspect retrieved context documents

In [ ]:
import pandas as pd

docs = result.get('context_docs', [])
print(f'Context documents retrieved: {len(docs)}')

if docs:
    meta_df = pd.DataFrame([d.metadata for d in docs])
    display(meta_df)

    # Show first table chunk if any
    tables = [d for d in docs if d.metadata.get('chunk_type') == 'table']
    if tables:
        from IPython.display import Markdown, display
        display(Markdown('### First Table Chunk\n\n' + tables[0].page_content))

## 5. Test a calculation query

In [ ]:
calc_result = run_query(
    query='Calculate JPMorgan Debt-to-Equity ratio for 2022.',
    thread_id=str(uuid.uuid4()),
    company='JPM',
    filing_type='10-K',
    year=2022,
    app=app,
)

print('Answer:')
print(calc_result['answer'])
print('\nExtracted metrics:', calc_result['extracted_metrics'])

## 6. Multi-turn conversation test (same thread_id)

In [ ]:
SESSION = str(uuid.uuid4())

turn_1 = run_query('What was MSFT revenue in 2023?', thread_id=SESSION, company='MSFT', year=2023, app=app)
print('Turn 1:', turn_1['answer'][:300])

turn_2 = run_query('What was their net income?', thread_id=SESSION, company='MSFT', year=2023, app=app)
print('\nTurn 2:', turn_2['answer'][:300])

## 7. Inspect agent_logs.db

In [ ]:
import sqlite3, pandas as pd, os

if os.path.exists(AGENT_LOGS_DB):
    with sqlite3.connect(AGENT_LOGS_DB) as conn:
        log_df = pd.read_sql_query('SELECT * FROM agent_queries ORDER BY timestamp DESC LIMIT 20', conn)
    display(log_df[['timestamp', 'company', 'filing_type', 'year', 'query', 'status', 'retry_count']])
else:
    print('agent_logs.db not found — run a query first.')

## 8. VRAM usage after full agent run

In [ ]:
from llm_engine import print_vram_usage
print_vram_usage()